# Week 2: Bimanual Manipulation with ALOHA — Diffusion Policy vs ACT

**Vizuara Robot Learning Bootcamp**

In this notebook, you will:
1. Understand the ALOHA bimanual manipulation platform and why it matters
2. Explore the Transfer Cube and Insertion datasets — what does bimanual coordination look like?
3. Understand Action Chunking with Transformers (ACT) — the policy ALOHA was designed for
4. Train ACT on Transfer Cube and achieve >80% success
5. Train Diffusion Policy on the same task and discover why it struggles
6. Diagnose the failure: the crop_shape problem and how to fix it
7. Compare both policies head-to-head: success rates, failure modes, inference speed
8. Visualize bimanual coordination patterns and multimodal action distributions

> **Hardware**: Google Colab with T4 GPU (free tier). ACT trains in ~2-3h; Diffusion Policy in ~3-4h.
>
> **Prerequisites**: Completion of Week 1 (PushT). You should understand DDPM, the 1D U-Net, FiLM conditioning, and SpatialSoftmax.

---

## Part 0: Setup & Installation

We install LeRobot with the ALOHA environment extra, which pulls in `gym-aloha` — a Gymnasium wrapper around the ALOHA simulation built on MuJoCo.

In [ ]:
# Install LeRobot with ALOHA environment support
# IMPORTANT: Install EGL for headless MuJoCo rendering on Colab (no display server)
!apt-get install -qq -y libegl1-mesa-dev libgl1-mesa-dev libgles2-mesa-dev > /dev/null 2>&1
!pip install -q "lerobot[aloha]" mediapy 2>&1 | tail -5

# Set MuJoCo to use EGL (headless GPU rendering) — MUST be before any MuJoCo import
import os
os.environ["MUJOCO_GL"] = "egl"

# Clear any corrupted dataset cache (prevents video decoding errors)
# If you hit "Could not push packet to decoder" errors, uncomment and run this:
# !rm -rf ~/.cache/huggingface/lerobot/lerobot/aloha_sim_transfer_cube_human
# !rm -rf ~/.cache/huggingface/lerobot/lerobot/aloha_sim_insertion_human

# Verify installations
import lerobot
import gym_aloha
print(f"LeRobot version: {lerobot.__version__}")
print(f"gym_aloha: OK")
print(f"MUJOCO_GL: {os.environ.get('MUJOCO_GL', 'not set')}")

In [ ]:
# Core imports
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from IPython.display import HTML
import mediapy as media
import gymnasium as gym
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

---
## Part 1: Meet ALOHA — A Low-cost Open-source Hardware for bimanual teleoperation

### Why ALOHA?

ALOHA (A Low-cost Open-source HArdware system) was introduced by [Zhao et al. (2023)](https://tonyzhaozh.github.io/aloha/) at Stanford. It consists of **two 6-DOF robot arms** with grippers, controlled via teleoperation.

**Why bimanual manipulation matters:**
- Many real-world tasks require two hands: opening a jar, folding laundry, handing objects
- Bimanual coordination is exponentially harder than single-arm: the action space doubles (14D vs 7D) and the two arms must avoid collisions while cooperating
- This is where policy learning really shines — hand-coding bimanual coordination is nightmarish

### The Two Tasks

| Task | Description | Difficulty | Why It's Interesting |
|------|-------------|-----------|---------------------|
| **Transfer Cube** | Left arm picks up a red cube, transfers it to the right arm's gripper | Medium | Tests handoff timing — both arms must coordinate the grasp-release precisely |
| **Insertion** | One arm holds a socket, the other inserts a peg | Hard | Tests precision alignment — sub-millimeter accuracy needed |

### Observation & Action Spaces

```
OBSERVATION:
  observation.images.top    : [480, 640, 3]  — overhead camera (the only camera in sim)
  observation.state         : [14]            — joint positions of both arms
                                                [left_waist, left_shoulder, left_elbow,
                                                 left_forearm_roll, left_wrist_angle,
                                                 left_wrist_rotate, left_gripper,
                                                 right_waist, right_shoulder, right_elbow,
                                                 right_forearm_roll, right_wrist_angle,
                                                 right_wrist_rotate, right_gripper]

ACTION:
  action                    : [14]            — target joint positions (same structure)
```

**Key differences from PushT (Week 1):**
| | PushT | ALOHA |
|---|-------|-------|
| Action dim | 2 (x, y) | 14 (7 joints × 2 arms) |
| Image size | 96×96 | 480×640 |
| State dim | 2 (agent x, y) | 14 (joint positions) |
| FPS | 10 | 50 |
| Demos | 205 | 50 |
| Coordination | Single agent | Two arms must cooperate |

In [ ]:
# Create the ALOHA Transfer Cube environment
env = gym.make(
    "gym_aloha/AlohaTransferCube-v0",
    obs_type="pixels_agent_pos",
    render_mode="rgb_array",
)

obs, info = env.reset(seed=42)

print("=== Transfer Cube: Observation Space ===")
for key, val in obs.items():
    if isinstance(val, np.ndarray):
        print(f"  {key:20s} shape={str(val.shape):15s} dtype={val.dtype}")

print(f"\n=== Action Space ===")
print(f"  shape : {env.action_space.shape}")   # (14,)
print(f"  low   : {env.action_space.low[:3]}...")
print(f"  high  : {env.action_space.high[:3]}...")

# Visualize the scene
frame = env.render()
plt.figure(figsize=(10, 7))
plt.imshow(frame)
plt.title("ALOHA Transfer Cube — Initial State\n(Left arm will pick red cube, hand to right arm)")
plt.axis('off')
plt.show()

env.close()

In [ ]:
# Also visualize the Insertion task
env_ins = gym.make(
    "gym_aloha/AlohaInsertion-v0",
    obs_type="pixels_agent_pos",
    render_mode="rgb_array",
)

obs_ins, _ = env_ins.reset(seed=42)
frame_ins = env_ins.render()

# Side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

env_tc = gym.make("gym_aloha/AlohaTransferCube-v0", obs_type="pixels_agent_pos", render_mode="rgb_array")
obs_tc, _ = env_tc.reset(seed=42)

axes[0].imshow(env_tc.render())
axes[0].set_title("Task 1: Transfer Cube\n(pick → hand over → receive)", fontsize=13)
axes[0].axis('off')

axes[1].imshow(frame_ins)
axes[1].set_title("Task 2: Insertion\n(grasp peg → align → insert into socket)", fontsize=13)
axes[1].axis('off')

plt.suptitle("ALOHA Bimanual Manipulation Tasks", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

env_tc.close()
env_ins.close()

In [ ]:
# Watch a random policy — bimanual random motion is even more chaotic than PushT
env = gym.make("gym_aloha/AlohaTransferCube-v0", obs_type="pixels_agent_pos", render_mode="rgb_array")
obs, info = env.reset(seed=42)

frames = [env.render()]
for step in range(100):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    if step % 2 == 0:
        frames.append(env.render())

env.close()
media.show_video(frames, fps=15, title="Random Policy on Transfer Cube (pure chaos)")
print("\nWith 14D actions and two arms, random motion is completely incoherent.")
print("The arms flail independently — no coordination at all.")

---
## Part 2: Exploring the Demonstration Datasets

The ALOHA simulation datasets contain 50 human teleoperation demonstrations each, recorded at 50 FPS.

**50 demos is very few!** This is a key challenge — can we learn bimanual coordination from just 50 trajectories?

- `lerobot/aloha_sim_transfer_cube_human` — 50 episodes of cube transfer
- `lerobot/aloha_sim_insertion_human` — 50 episodes of peg insertion

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata

# Load metadata for both datasets
meta_transfer = LeRobotDatasetMetadata("lerobot/aloha_sim_transfer_cube_human")
meta_insertion = LeRobotDatasetMetadata("lerobot/aloha_sim_insertion_human")

print("=== Transfer Cube Dataset ===")
print(f"  FPS: {meta_transfer.fps}")
print(f"  Total episodes: {meta_transfer.total_episodes}")
print(f"  Total frames: {meta_transfer.total_frames}")
print(f"\n  Features:")
for name, feat in meta_transfer.features.items():
    print(f"    {name}: {feat}")

print(f"\n=== Insertion Dataset ===")
print(f"  FPS: {meta_insertion.fps}")
print(f"  Total episodes: {meta_insertion.total_episodes}")
print(f"  Total frames: {meta_insertion.total_frames}")

In [ ]:
# Load the Transfer Cube dataset (we'll focus on this task first)
# Note: at 50 FPS, one frame = 0.02 seconds

dataset_transfer = LeRobotDataset("lerobot/aloha_sim_transfer_cube_human")

print(f"Dataset size: {len(dataset_transfer)} samples")
print(f"Average episode length: {len(dataset_transfer) / meta_transfer.total_episodes:.0f} frames")
print(f"Average episode duration: {len(dataset_transfer) / meta_transfer.total_episodes / meta_transfer.fps:.1f} seconds")

# Inspect a single sample
sample = dataset_transfer[0]
print(f"\n=== Single Sample ===")
for key, val in sample.items():
    if isinstance(val, torch.Tensor):
        print(f"  {key:30s} shape={str(val.shape):15s} dtype={val.dtype}")
    else:
        print(f"  {key:30s} = {val}")

In [ ]:
# Visualize frames from a single episode
# Show the progression: approach → grasp → lift → handoff → receive

# Find the start and end of episode 0
ep_indices = [i for i in range(len(dataset_transfer))
              if dataset_transfer[i]['episode_index'].item() == 0]
ep_length = len(ep_indices)
print(f"Episode 0 length: {ep_length} frames ({ep_length/meta_transfer.fps:.1f}s)")

# Sample 8 evenly spaced frames
frame_indices = np.linspace(ep_indices[0], ep_indices[-1], 8, dtype=int)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, idx in enumerate(frame_indices):
    s = dataset_transfer[idx]
    img = s['observation.images.top'].permute(1, 2, 0).numpy()
    # Normalize to [0, 1] if needed
    if img.max() > 1.0:
        img = img / 255.0
    img = np.clip(img, 0, 1)

    t_sec = (idx - ep_indices[0]) / meta_transfer.fps
    axes[i].imshow(img)
    axes[i].set_title(f"t = {t_sec:.1f}s (frame {idx - ep_indices[0]})", fontsize=11)
    axes[i].axis('off')

plt.suptitle("Episode 0: Transfer Cube Demonstration\n(approach → grasp → lift → handoff → receive)",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Analyze the 14D action space — what does bimanual coordination look like?

joint_names = [
    'L_waist', 'L_shoulder', 'L_elbow',
    'L_forearm', 'L_wrist_ang', 'L_wrist_rot', 'L_gripper',
    'R_waist', 'R_shoulder', 'R_elbow',
    'R_forearm', 'R_wrist_ang', 'R_wrist_rot', 'R_gripper',
]

# Collect all actions from episode 0
ep_actions = []
ep_states = []
for idx in ep_indices:
    s = dataset_transfer[idx]
    ep_actions.append(s['action'].numpy())
    ep_states.append(s['observation.state'].numpy())

ep_actions = np.array(ep_actions)  # [T, 14]
ep_states = np.array(ep_states)    # [T, 14]

print(f"Episode actions shape: {ep_actions.shape}")
print(f"Episode states shape: {ep_states.shape}")

# Plot all 14 joint trajectories
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
t = np.arange(len(ep_actions)) / meta_transfer.fps  # Time in seconds

# Left arm (joints 0-6)
for j in range(7):
    axes[0].plot(t, ep_actions[:, j], label=joint_names[j], linewidth=1.5)
axes[0].set_title('Left Arm Joint Commands (7 DOF)', fontsize=13)
axes[0].set_ylabel('Joint position (rad)')
axes[0].legend(loc='upper right', fontsize=9)
axes[0].grid(True, alpha=0.3)

# Right arm (joints 7-13)
for j in range(7, 14):
    axes[1].plot(t, ep_actions[:, j], label=joint_names[j], linewidth=1.5)
axes[1].set_title('Right Arm Joint Commands (7 DOF)', fontsize=13)
axes[1].set_xlabel('Time (seconds)')
axes[1].set_ylabel('Joint position (rad)')
axes[1].legend(loc='upper right', fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Bimanual Coordination: Joint Trajectories During Cube Transfer', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nNotice: Both arms move simultaneously — the left arm approaches and grasps")
print("while the right arm positions itself to receive. The gripper signals")
print("(L_gripper, R_gripper) show the critical grasp/release timing.")

In [ ]:
# Zoom in on the gripper coordination — this is the critical handoff moment

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(t, ep_actions[:, 6], 'b-', linewidth=2.5, label='Left Gripper (source)')
ax.plot(t, ep_actions[:, 13], 'r-', linewidth=2.5, label='Right Gripper (receiver)')

ax.set_xlabel('Time (seconds)', fontsize=12)
ax.set_ylabel('Gripper position', fontsize=12)
ax.set_title('Gripper Coordination During Cube Transfer\n'
             'Left must OPEN at the exact moment Right CLOSES',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

# Annotate phases
ax.axvspan(0, t[-1]*0.3, alpha=0.1, color='blue', label='_Left approaches')
ax.axvspan(t[-1]*0.3, t[-1]*0.6, alpha=0.1, color='green', label='_Handoff zone')
ax.axvspan(t[-1]*0.6, t[-1], alpha=0.1, color='red', label='_Right receives')

plt.tight_layout()
plt.show()

print("This coordination is what makes bimanual manipulation hard:")
print("- If the left arm opens too early → cube drops")
print("- If the right arm closes too late → cube slips through")
print("- The policy must learn this precise temporal coupling from only 50 demos!")

In [ ]:
# Cross-episode analysis: how consistent are the demonstrations?

# Collect gripper trajectories from all 50 episodes
all_left_grippers = []
all_right_grippers = []

for ep_id in range(min(meta_transfer.total_episodes, 20)):  # First 20 episodes
    ep_idx = [i for i in range(len(dataset_transfer))
              if dataset_transfer[i]['episode_index'].item() == ep_id]
    if len(ep_idx) == 0:
        continue
    grips = []
    for idx in ep_idx:
        s = dataset_transfer[idx]
        grips.append(s['action'].numpy())
    grips = np.array(grips)
    all_left_grippers.append(grips[:, 6])
    all_right_grippers.append(grips[:, 13])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for grips in all_left_grippers:
    t_norm = np.linspace(0, 1, len(grips))  # Normalize time to [0, 1]
    axes[0].plot(t_norm, grips, alpha=0.3, color='blue', linewidth=1)
axes[0].set_title('Left Gripper Across 20 Episodes', fontsize=12)
axes[0].set_xlabel('Normalized time')
axes[0].set_ylabel('Gripper position')
axes[0].grid(True, alpha=0.3)

for grips in all_right_grippers:
    t_norm = np.linspace(0, 1, len(grips))
    axes[1].plot(t_norm, grips, alpha=0.3, color='red', linewidth=1)
axes[1].set_title('Right Gripper Across 20 Episodes', fontsize=12)
axes[1].set_xlabel('Normalized time')
axes[1].set_ylabel('Gripper position')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Demonstration Variability — Each line is a different episode',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Some demonstrations are faster, some slower.")
print("Some approach from slightly different angles.")
print("The policy must learn the common structure despite this variation.")

---
## Part 3: Understanding ACT — Action Chunking with Transformers

ACT was specifically designed for ALOHA by the original ALOHA authors. It's the **recommended starting policy** for bimanual manipulation.

### Why ACT before Diffusion Policy?

| Aspect | ACT | Diffusion Policy |
|--------|-----|------------------|
| Designed for | ALOHA bimanual tasks | General manipulation |
| Success rate (transfer cube) | **~83%** at 80k steps | 2-20% (with default settings) |
| Training time (A100) | ~1.5 hours | Much longer |
| Data efficiency | Works with 50 demos | Needs 100+ demos |
| Inference speed | Single forward pass | 100 denoising steps |
| Multimodality | Via VAE latent | Via diffusion sampling |

### ACT Architecture Overview

```
ENCODER (Condition Extraction):
  RGB image (480×640×3)  ──► ResNet18 (ImageNet pretrained) ──► Feature tokens
  Joint state [14]       ──► Linear projection ──────────────► State token
  [VAE latent z ~ N(0,I)] ──► Linear projection ──────────────► Style token
       │
       ▼
  Transformer Encoder (4 layers, 512d, 8 heads)
       │
       ▼
  Encoder output = conditioning memory

DECODER (Action Prediction):
  100 learnable action queries ──► Transformer Decoder (1 layer)
                                   cross-attends to encoder output
       │
       ▼
  100 action predictions [each 14D]
```

### Key Differences from Diffusion Policy

1. **Chunk size = 100**: ACT predicts 100 future actions at once (vs 16 for Diffusion Policy), and **executes all 100** (vs only 8)
2. **Single forward pass**: No iterative denoising — one pass through the transformer gives all actions
3. **VAE for multimodality**: Instead of starting from random noise (diffusion), ACT uses a VAE latent variable `z` to capture action diversity
4. **ImageNet pretrained backbone**: ACT uses pretrained ResNet18 weights; Diffusion Policy trains from scratch by default
5. **MEAN_STD normalization**: ACT normalizes everything with mean/std; Diffusion Policy uses MIN_MAX for state/actions

In [ ]:
from lerobot.policies.act.configuration_act import ACTConfig
from lerobot.policies.act.modeling_act import ACTPolicy
from lerobot.configs.types import FeatureType
from lerobot.datasets.utils import dataset_to_policy_features

# Build ACT config for the ALOHA transfer cube task
features = dataset_to_policy_features(meta_transfer.features)
output_features = {k: ft for k, ft in features.items() if ft.type is FeatureType.ACTION}
input_features = {k: ft for k, ft in features.items() if k not in output_features}

act_cfg = ACTConfig(input_features=input_features, output_features=output_features)

print("=== ACT Configuration ===")
print(f"  n_obs_steps          = {act_cfg.n_obs_steps}")
print(f"  chunk_size           = {act_cfg.chunk_size}         # Predict 100 actions at once!")
print(f"  n_action_steps       = {act_cfg.n_action_steps}       # Execute ALL 100")
print(f"  vision_backbone      = {act_cfg.vision_backbone}")
print(f"  pretrained_weights   = {act_cfg.pretrained_backbone_weights}")
print(f"  dim_model            = {act_cfg.dim_model}")
print(f"  n_heads              = {act_cfg.n_heads}")
print(f"  n_encoder_layers     = {act_cfg.n_encoder_layers}")
print(f"  n_decoder_layers     = {act_cfg.n_decoder_layers}")
print(f"  use_vae              = {act_cfg.use_vae}")
print(f"  latent_dim           = {act_cfg.latent_dim}")
print(f"  kl_weight            = {act_cfg.kl_weight}")
print(f"  dropout              = {act_cfg.dropout}")
print(f"  optimizer_lr         = {act_cfg.optimizer_lr}")

In [ ]:
# Instantiate ACT and inspect the architecture
act_policy = ACTPolicy(act_cfg)
act_policy.to(device)

total_params = sum(p.numel() for p in act_policy.parameters())
trainable_params = sum(p.numel() for p in act_policy.parameters() if p.requires_grad)
print(f"\nACT Total parameters: {total_params:,} ({total_params/1e6:.1f}M)")
print(f"ACT Trainable parameters: {trainable_params:,}")

# Compare with Diffusion Policy
from lerobot.policies.diffusion.configuration_diffusion import DiffusionConfig
from lerobot.policies.diffusion.modeling_diffusion import DiffusionPolicy

diff_cfg = DiffusionConfig(input_features=input_features, output_features=output_features)
diff_policy = DiffusionPolicy(diff_cfg)
diff_params = sum(p.numel() for p in diff_policy.parameters())
print(f"\nDiffusion Policy parameters: {diff_params:,} ({diff_params/1e6:.1f}M)")
print(f"\nACT is {'smaller' if total_params < diff_params else 'larger'} by {abs(total_params-diff_params):,} params")

### The VAE in ACT — How It Handles Multimodality

In Week 1, we saw that Diffusion Policy handles multimodal action distributions by starting from different random noise and denoising to different valid trajectories.

ACT takes a different approach: a **Variational Autoencoder (VAE)**.

**During training:**
1. The **VAE encoder** sees the ground-truth actions and compresses them into a latent vector `z ~ N(μ, σ²)`
2. This `z` is passed to the decoder along with observations
3. The decoder reconstructs the actions conditioned on both observations AND `z`
4. KL loss keeps `z` close to N(0, I)

**During inference:**
1. We don't have ground-truth actions, so we sample `z ~ N(0, I)`
2. Different `z` samples produce different valid action trajectories
3. This is how ACT captures multiple valid solutions

```
TRAINING:                              INFERENCE:
GT actions ──► VAE encoder ──► z       z ~ N(0, I)  (random sample)
                   │                        │
                   ▼                        ▼
              Decoder(obs, z)           Decoder(obs, z)
                   │                        │
                   ▼                        ▼
           Reconstructed actions     Predicted actions
```

---
## Part 4: Training ACT on Transfer Cube

ACT is the recommended starting policy for ALOHA. It trains faster and achieves higher success rates.

**Training plan:**
- Dataset: `lerobot/aloha_sim_transfer_cube_human` (50 demos, 50 FPS)
- Steps: 50,000 (achieves ~80% on A100; on T4 we'll use fewer steps for faster iteration)
- Batch size: 8 (memory-friendly for T4)
- Optimizer: Adam, lr=1e-5 (ACT uses a lower learning rate than Diffusion Policy)

In [ ]:
from lerobot.policies.factory import make_pre_post_processors

# Helper to convert delta indices to timestamps
def make_delta_timestamps(delta_indices, fps):
    if delta_indices is None:
        return [0.0]
    return [i / fps for i in delta_indices]

# Fresh ACT policy for training
act_cfg = ACTConfig(input_features=input_features, output_features=output_features)
act_policy = ACTPolicy(act_cfg)
act_policy.train()
act_policy.to(device)

act_preprocessor, act_postprocessor = make_pre_post_processors(
    act_cfg, dataset_stats=meta_transfer.stats
)

# CRITICAL: delta_timestamps setup for ACT
# - action: needs chunk_size (100) future timesteps
# - image keys: need [0.0] for video decoding (squeeze removes the extra dim)
# - observation.state: must NOT be included (would add an unwanted time dimension)
#
# ACT's config tells us exactly what to use:
#   cfg.action_delta_indices  = [0, 1, ..., 99]  (chunk_size steps)
#   cfg.observation_delta_indices = None           (no temporal stacking for obs)

delta_timestamps_act = {
    "action": make_delta_timestamps(act_cfg.action_delta_indices, meta_transfer.fps),
}
# Add image keys only (NOT observation.state)
delta_timestamps_act |= {
    k: make_delta_timestamps(act_cfg.observation_delta_indices, meta_transfer.fps)
    for k in act_cfg.image_features
}

print(f"delta_timestamps keys: {list(delta_timestamps_act.keys())}")
print(f"  action: {len(delta_timestamps_act['action'])} timesteps")
for k in act_cfg.image_features:
    print(f"  {k}: {delta_timestamps_act[k]}")

dataset_act = LeRobotDataset(
    "lerobot/aloha_sim_transfer_cube_human",
    delta_timestamps=delta_timestamps_act,
)

print(f"\nACT dataset size: {len(dataset_act)} samples")
sample = dataset_act[0]
for key, val in sample.items():
    if isinstance(val, torch.Tensor):
        print(f"  {key:30s} shape={str(val.shape):15s}")
    else:
        print(f"  {key:30s} = {val}")

In [ ]:
# Training loop for ACT
ACT_TRAINING_STEPS = 30000  # Use 50000-80000 for best results
ACT_BATCH_SIZE = 8          # Increase to 16 if GPU memory allows
ACT_LR = 1e-5
LOG_FREQ = 500

act_optimizer = torch.optim.Adam(
    act_policy.parameters(),
    lr=ACT_LR,
    weight_decay=act_cfg.optimizer_weight_decay,
)

act_dataloader = torch.utils.data.DataLoader(
    dataset_act,
    batch_size=ACT_BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True,
)

print(f"Training ACT for {ACT_TRAINING_STEPS} steps")
print(f"Batch size: {ACT_BATCH_SIZE}, LR: {ACT_LR}")
print(f"Batches per epoch: {len(act_dataloader)}")
print(f"\nStarting training...\n")

act_losses = []
step = 0
done = False
start_time = time.time()

while not done:
    for batch in act_dataloader:
        batch = act_preprocessor(batch)

        loss, info = act_policy.forward(batch)
        loss.backward()
        act_optimizer.step()
        act_optimizer.zero_grad()

        act_losses.append(loss.item())

        if step % LOG_FREQ == 0:
            elapsed = time.time() - start_time
            avg_loss = np.mean(act_losses[-LOG_FREQ:]) if len(act_losses) >= LOG_FREQ else np.mean(act_losses)
            steps_per_sec = (step + 1) / elapsed
            eta = (ACT_TRAINING_STEPS - step) / max(steps_per_sec, 0.01)
            print(f"Step {step:6d}/{ACT_TRAINING_STEPS} | Loss: {loss.item():.4f} | "
                  f"Avg: {avg_loss:.4f} | {steps_per_sec:.1f} it/s | ETA: {eta/60:.1f}min")

        step += 1
        if step >= ACT_TRAINING_STEPS:
            done = True
            break

total_time = time.time() - start_time
print(f"\nACT training complete! {ACT_TRAINING_STEPS} steps in {total_time/60:.1f} minutes")
print(f"Final loss: {act_losses[-1]:.4f}")

In [ ]:
# Plot ACT training curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

window = min(200, len(act_losses) // 10)

axes[0].plot(act_losses, alpha=0.2, color='blue')
if window > 1:
    smoothed = np.convolve(act_losses, np.ones(window)/window, mode='valid')
    axes[0].plot(range(window-1, len(act_losses)), smoothed, 'r-', linewidth=2, label=f'Smoothed (w={window})')
axes[0].set_xlabel('Training step')
axes[0].set_ylabel('Loss (L1 + KL)')
axes[0].set_title('ACT Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].semilogy(act_losses, alpha=0.2, color='blue')
if window > 1:
    axes[1].semilogy(range(window-1, len(act_losses)), smoothed, 'r-', linewidth=2)
axes[1].set_xlabel('Training step')
axes[1].set_ylabel('Loss (log scale)')
axes[1].set_title('ACT Training Loss (log)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Save ACT checkpoint
act_output_dir = Path("outputs/act_aloha_transfer")
act_output_dir.mkdir(parents=True, exist_ok=True)

act_policy.save_pretrained(act_output_dir)
act_preprocessor.save_pretrained(act_output_dir)
act_postprocessor.save_pretrained(act_output_dir)
print(f"ACT model saved to {act_output_dir}")

---
## Part 5: Evaluating ACT on Transfer Cube

Let's run the trained ACT policy in the ALOHA simulation and measure success rate.

In [ ]:
def evaluate_aloha_policy(policy, preprocessor, postprocessor, task="AlohaTransferCube-v0",
                          n_episodes=10, max_steps=400, seed=42):
    """
    Evaluate a policy on an ALOHA task.
    Returns rewards, success flags, and recorded frames for each episode.
    """
    env = gym.make(
        f"gym_aloha/{task}",
        obs_type="pixels_agent_pos",
        render_mode="rgb_array",
    )

    all_rewards = []
    all_frames = []
    successes = []

    for ep in range(n_episodes):
        obs, info = env.reset(seed=seed + ep)
        policy.reset()

        frames = [env.render()]
        total_reward = 0
        max_reward = 0

        for step_i in range(max_steps):
            # Build observation dict
            state = torch.from_numpy(obs['agent_pos']).float().unsqueeze(0).to(device)

            # Handle image: may be in 'pixels' dict with camera names
            if 'pixels' in obs and isinstance(obs['pixels'], dict):
                img = obs['pixels']['top']
            elif 'pixels' in obs:
                img = obs['pixels']
            else:
                img = obs.get('top', obs.get('image', None))

            image = torch.from_numpy(img).float().permute(2, 0, 1).unsqueeze(0).to(device) / 255.0

            obs_dict = {
                'observation.state': state,
                'observation.images.top': image,
            }
            obs_dict = preprocessor(obs_dict)

            with torch.no_grad():
                action = policy.select_action(obs_dict)

            action = postprocessor(action)
            action_np = action.squeeze(0).cpu().numpy()

            obs, reward, terminated, truncated, info = env.step(action_np)
            total_reward += reward
            max_reward = max(max_reward, reward)

            if step_i % 5 == 0:
                frames.append(env.render())

            if terminated or truncated:
                break

        # For transfer cube: reward >= 4 means full success
        success = max_reward >= 4.0
        successes.append(success)
        all_rewards.append(total_reward)
        all_frames.append(frames)

        status = "SUCCESS" if success else "FAIL"
        print(f"  Ep {ep+1:2d}: reward={total_reward:6.1f} | max_r={max_reward:.0f} | [{status}]")

    env.close()

    success_rate = np.mean(successes) * 100
    print(f"\nSuccess rate: {success_rate:.0f}% ({sum(successes)}/{n_episodes})")
    print(f"Mean reward: {np.mean(all_rewards):.1f} ± {np.std(all_rewards):.1f}")

    return all_rewards, successes, all_frames

In [ ]:
# Evaluate ACT
print("=== Evaluating ACT on Transfer Cube ===")
act_policy.eval()
act_rewards, act_successes, act_frames = evaluate_aloha_policy(
    act_policy, act_preprocessor, act_postprocessor,
    n_episodes=10, max_steps=400,
)

In [ ]:
# Show the best ACT episode
best_ep = np.argmax(act_rewards)
print(f"Showing best ACT episode ({best_ep+1}) with reward {act_rewards[best_ep]:.1f}")
media.show_video(act_frames[best_ep], fps=15, title=f"ACT — Transfer Cube (reward={act_rewards[best_ep]:.1f})")

In [ ]:
# If there's a failure, show it too for comparison
fail_eps = [i for i, s in enumerate(act_successes) if not s]
if fail_eps:
    fail_ep = fail_eps[0]
    print(f"Showing failed ACT episode ({fail_ep+1}) with reward {act_rewards[fail_ep]:.1f}")
    media.show_video(act_frames[fail_ep], fps=15,
                     title=f"ACT — FAILURE (reward={act_rewards[fail_ep]:.1f})")
    print("\nCommon failure modes:")
    print("- Grasp timing: left arm doesn't grip cube firmly")
    print("- Handoff desync: arms don't meet at the right position")
    print("- Drop: cube slips during transfer")
else:
    print("All episodes succeeded! Try evaluating with more episodes.")

---
## Part 6: Training Diffusion Policy on Transfer Cube

Now let's train the same task with Diffusion Policy and see how it compares.

### The Crop Shape Problem

**Critical issue** documented in [LeRobot GitHub Issue #502](https://github.com/huggingface/lerobot/issues/502):

Diffusion Policy's default `crop_shape = [84, 84]` with `crop_is_random = True` was designed for PushT (96×96 images). When applied to ALOHA's 480×640 images, it randomly crops a tiny 84×84 patch — **destroying most of the spatial information the policy needs**.

This is why "out of the box" Diffusion Policy only achieves **2-6% success** on ALOHA.

**The fix**: Either disable cropping or increase `crop_shape` to something reasonable.

We'll train **two versions** to demonstrate this:
1. **Naive** (default crop): to see the failure
2. **Fixed** (no crop / large crop): to see the improvement

This is a valuable lesson: **hyperparameters that work for one task can catastrophically fail on another.**

In [ ]:
from lerobot.policies.diffusion.configuration_diffusion import DiffusionConfig
from lerobot.policies.diffusion.modeling_diffusion import DiffusionPolicy

# Build Diffusion Policy config first to get its delta indices
diff_cfg_temp = DiffusionConfig(input_features=input_features, output_features=output_features)

# Use the config's own delta index properties (same pattern as ACT)
# Diffusion Policy uses:
#   observation_delta_indices = based on n_obs_steps (e.g., [-1, 0] for 2 obs steps)
#   action_delta_indices = based on horizon (e.g., [-1, 0, 1, ..., 14] for horizon=16)

delta_timestamps_diff = {
    "action": make_delta_timestamps(diff_cfg_temp.action_delta_indices, meta_transfer.fps),
}
# Add observation keys using the config's observation_delta_indices
delta_timestamps_diff |= {
    k: make_delta_timestamps(diff_cfg_temp.observation_delta_indices, meta_transfer.fps)
    for k in diff_cfg_temp.image_features
}
# Add state features with observation_delta_indices too
delta_timestamps_diff |= {
    k: make_delta_timestamps(diff_cfg_temp.observation_delta_indices, meta_transfer.fps)
    for k in diff_cfg_temp.state_features
}

print(f"Diffusion delta_timestamps:")
for k, v in delta_timestamps_diff.items():
    print(f"  {k}: {len(v)} timesteps — {v[:3]}{'...' if len(v) > 3 else ''}")

dataset_diff = LeRobotDataset(
    "lerobot/aloha_sim_transfer_cube_human",
    delta_timestamps=delta_timestamps_diff,
)

print(f"\nDiffusion dataset size: {len(dataset_diff)} samples")
sample = dataset_diff[0]
for key, val in sample.items():
    if isinstance(val, torch.Tensor):
        print(f"  {key:30s} shape={str(val.shape):15s}")
    else:
        print(f"  {key:30s} = {val}")

In [ ]:
# Visualize: what does the default 84x84 crop look like on a 480x640 image?

sample = dataset_diff[100]
img = sample['observation.images.top'][-1].permute(1, 2, 0).numpy()  # Last frame
if img.max() > 1.0:
    img = img / 255.0
img = np.clip(img, 0, 1)
h, w = img.shape[:2]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Full image
axes[0].imshow(img)
axes[0].set_title(f'Full image ({h}×{w})', fontsize=12)
axes[0].axis('off')

# Show random 84x84 crop (the DEFAULT — this is the problem!)
axes[1].imshow(img)
# Draw several random crop boxes to show how little they cover
for _ in range(5):
    cy = np.random.randint(0, h - 84)
    cx = np.random.randint(0, w - 84)
    rect = plt.Rectangle((cx, cy), 84, 84, linewidth=2, edgecolor='red', facecolor='none')
    axes[1].add_patch(rect)
axes[1].set_title(f'Default crop: 84×84 (RED boxes)\n'
                  f'Covers only {84*84/(h*w)*100:.1f}% of image!',
                  fontsize=12, color='red')
axes[1].axis('off')

# Show fixed crop (440x560)
axes[2].imshow(img)
cy = (h - 440) // 2
cx = (w - 560) // 2
rect = plt.Rectangle((cx, cy), 560, 440, linewidth=3, edgecolor='green', facecolor='none')
axes[2].add_patch(rect)
axes[2].set_title(f'Fixed crop: 440×560 (GREEN box)\n'
                  f'Covers {440*560/(h*w)*100:.1f}% of image',
                  fontsize=12, color='green')
axes[2].axis('off')

plt.suptitle('The Crop Shape Problem: Why Default Diffusion Policy Fails on ALOHA',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Default 84×84 crop = {84*84/(h*w)*100:.1f}% of the 480×640 image")
print(f"The random crop almost always MISSES the robot arms and the cube!")
print(f"The policy is essentially training on random background patches.")

In [ ]:
# Train Diffusion Policy with the FIXED crop shape
# We disable the crop entirely by not setting crop_shape (or set a large one)

diff_cfg_fixed = DiffusionConfig(
    input_features=input_features,
    output_features=output_features,
    # THE FIX: reasonable crop that preserves spatial information
    crop_shape=(440, 560),
    crop_is_random=False,    # Center crop for consistency
    # Keep other defaults
    n_obs_steps=2,
    horizon=16,
    n_action_steps=8,
    num_train_timesteps=100,
)

diff_policy_fixed = DiffusionPolicy(diff_cfg_fixed)
diff_policy_fixed.train()
diff_policy_fixed.to(device)

diff_preprocessor, diff_postprocessor = make_pre_post_processors(
    diff_cfg_fixed, dataset_stats=meta_transfer.stats
)

print("=== Fixed Diffusion Policy Config ===")
print(f"  crop_shape    = {diff_cfg_fixed.crop_shape}")
print(f"  crop_is_random = {diff_cfg_fixed.crop_is_random}")
print(f"  n_obs_steps   = {diff_cfg_fixed.n_obs_steps}")
print(f"  horizon       = {diff_cfg_fixed.horizon}")
print(f"  n_action_steps = {diff_cfg_fixed.n_action_steps}")
print(f"  num_train_ts  = {diff_cfg_fixed.num_train_timesteps}")

n_params = sum(p.numel() for p in diff_policy_fixed.parameters())
print(f"  Parameters    = {n_params:,}")

In [ ]:
# Training loop for Diffusion Policy (fixed crop)
DIFF_TRAINING_STEPS = 30000  # Use 100000-200000 for best results
DIFF_BATCH_SIZE = 16         # Smaller images after crop = more batch room
DIFF_LR = 1e-4
LOG_FREQ = 500

diff_optimizer = torch.optim.Adam(
    diff_policy_fixed.parameters(),
    lr=DIFF_LR,
    betas=(0.95, 0.999),
    eps=1e-8,
    weight_decay=1e-6,
)

diff_dataloader = torch.utils.data.DataLoader(
    dataset_diff,
    batch_size=DIFF_BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True,
)

print(f"Training Diffusion Policy (fixed crop) for {DIFF_TRAINING_STEPS} steps")
print(f"Batch size: {DIFF_BATCH_SIZE}, LR: {DIFF_LR}")
print(f"\nStarting training...\n")

diff_losses = []
step = 0
done = False
start_time = time.time()

while not done:
    for batch in diff_dataloader:
        batch = diff_preprocessor(batch)

        loss, _ = diff_policy_fixed.forward(batch)
        loss.backward()
        diff_optimizer.step()
        diff_optimizer.zero_grad()

        diff_losses.append(loss.item())

        if step % LOG_FREQ == 0:
            elapsed = time.time() - start_time
            avg_loss = np.mean(diff_losses[-LOG_FREQ:]) if len(diff_losses) >= LOG_FREQ else np.mean(diff_losses)
            steps_per_sec = (step + 1) / elapsed
            eta = (DIFF_TRAINING_STEPS - step) / max(steps_per_sec, 0.01)
            print(f"Step {step:6d}/{DIFF_TRAINING_STEPS} | Loss: {loss.item():.4f} | "
                  f"Avg: {avg_loss:.4f} | {steps_per_sec:.1f} it/s | ETA: {eta/60:.1f}min")

        step += 1
        if step >= DIFF_TRAINING_STEPS:
            done = True
            break

total_time = time.time() - start_time
print(f"\nDiffusion training complete! {DIFF_TRAINING_STEPS} steps in {total_time/60:.1f} minutes")
print(f"Final loss: {diff_losses[-1]:.4f}")

In [ ]:
# Save Diffusion Policy checkpoint
diff_output_dir = Path("outputs/diffusion_aloha_transfer_fixed")
diff_output_dir.mkdir(parents=True, exist_ok=True)

diff_policy_fixed.save_pretrained(diff_output_dir)
diff_preprocessor.save_pretrained(diff_output_dir)
diff_postprocessor.save_pretrained(diff_output_dir)
print(f"Diffusion Policy saved to {diff_output_dir}")

In [ ]:
# Evaluate Diffusion Policy
print("=== Evaluating Diffusion Policy (fixed crop) on Transfer Cube ===")
diff_policy_fixed.eval()
diff_rewards, diff_successes, diff_frames = evaluate_aloha_policy(
    diff_policy_fixed, diff_preprocessor, diff_postprocessor,
    n_episodes=10, max_steps=400,
)

In [ ]:
# Show best Diffusion Policy episode
best_diff = np.argmax(diff_rewards)
print(f"Showing best Diffusion Policy episode ({best_diff+1})")
media.show_video(diff_frames[best_diff], fps=15,
                 title=f"Diffusion Policy — Transfer Cube (reward={diff_rewards[best_diff]:.1f})")

---
## Part 7: Head-to-Head Comparison — ACT vs Diffusion Policy

Now we have both policies trained. Let's do a rigorous comparison across multiple dimensions.

In [ ]:
# Comparison: Success rates
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Bar chart of success rates
act_sr = np.mean(act_successes) * 100
diff_sr = np.mean(diff_successes) * 100

bars = axes[0].bar(['ACT', 'Diffusion\n(fixed crop)'], [act_sr, diff_sr],
                   color=['steelblue', 'coral'], width=0.5)
axes[0].set_ylabel('Success Rate (%)')
axes[0].set_title('Success Rate Comparison')
axes[0].set_ylim(0, 100)
for bar, val in zip(bars, [act_sr, diff_sr]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                f'{val:.0f}%', ha='center', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

# Reward distributions
axes[1].boxplot([act_rewards, diff_rewards], labels=['ACT', 'Diffusion'])
axes[1].set_ylabel('Total Reward')
axes[1].set_title('Reward Distribution')
axes[1].grid(True, alpha=0.3)

# Training curves comparison
window = 200
if len(act_losses) > window:
    act_smooth = np.convolve(act_losses, np.ones(window)/window, mode='valid')
    axes[2].semilogy(act_smooth, label='ACT', linewidth=2, color='steelblue')
if len(diff_losses) > window:
    diff_smooth = np.convolve(diff_losses, np.ones(window)/window, mode='valid')
    axes[2].semilogy(diff_smooth, label='Diffusion (fixed)', linewidth=2, color='coral')
axes[2].set_xlabel('Training step')
axes[2].set_ylabel('Loss (log)')
axes[2].set_title('Training Curves')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('ACT vs Diffusion Policy on ALOHA Transfer Cube', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Inference speed comparison
print("=== Inference Speed Comparison ===")

# Create a dummy observation
dummy_state = torch.randn(1, 14).to(device)
dummy_image = torch.randn(1, 3, 480, 640).to(device)

dummy_obs = {
    'observation.state': dummy_state,
    'observation.images.top': dummy_image,
}

# Warm up
act_policy.eval()
act_policy.reset()
diff_policy_fixed.eval()
diff_policy_fixed.reset()

# Time ACT inference
n_runs = 10

act_policy.reset()
act_obs = act_preprocessor(dummy_obs)
with torch.no_grad():
    _ = act_policy.select_action(act_obs)  # Warmup

act_policy.reset()
start = time.time()
with torch.no_grad():
    for _ in range(n_runs):
        act_policy.reset()
        _ = act_policy.select_action(act_obs)
act_time = (time.time() - start) / n_runs * 1000

# Time Diffusion Policy inference
diff_policy_fixed.reset()
diff_obs = diff_preprocessor(dummy_obs)
with torch.no_grad():
    _ = diff_policy_fixed.select_action(diff_obs)  # Warmup

diff_policy_fixed.reset()
start = time.time()
with torch.no_grad():
    for _ in range(n_runs):
        diff_policy_fixed.reset()
        _ = diff_policy_fixed.select_action(diff_obs)
diff_time = (time.time() - start) / n_runs * 1000

print(f"ACT inference:       {act_time:6.1f} ms  (single forward pass)")
print(f"Diffusion inference: {diff_time:6.1f} ms  (100 denoising steps)")
print(f"\nDiffusion is {diff_time/act_time:.1f}x slower than ACT")
print(f"\nAt 50 FPS (20ms/frame), ACT is {'real-time capable' if act_time < 20 else 'too slow for real-time'}")
print(f"Diffusion Policy is {'real-time capable' if diff_time < 20 else 'too slow for real-time (but can use action chunking to amortize)'}")

In [ ]:
# Summary comparison table
print("\n" + "="*70)
print("      ACT vs DIFFUSION POLICY — ALOHA TRANSFER CUBE")
print("="*70)
print(f"{'Metric':<30s} {'ACT':>15s} {'Diffusion':>15s}")
print("-"*70)
print(f"{'Success Rate':<30s} {act_sr:>14.0f}% {diff_sr:>14.0f}%")
print(f"{'Mean Reward':<30s} {np.mean(act_rewards):>15.1f} {np.mean(diff_rewards):>15.1f}")
print(f"{'Training Steps':<30s} {ACT_TRAINING_STEPS:>15,d} {DIFF_TRAINING_STEPS:>15,d}")
print(f"{'Inference Time (ms)':<30s} {act_time:>15.1f} {diff_time:>15.1f}")
print(f"{'Action Chunk Size':<30s} {act_cfg.chunk_size:>15d} {diff_cfg_fixed.horizon:>15d}")
print(f"{'Actions Executed':<30s} {act_cfg.n_action_steps:>15d} {diff_cfg_fixed.n_action_steps:>15d}")
print(f"{'Parameters':<30s} {sum(p.numel() for p in act_policy.parameters()):>15,d} {sum(p.numel() for p in diff_policy_fixed.parameters()):>15,d}")
print(f"{'Multimodality Mechanism':<30s} {'VAE latent z':>15s} {'Noise sampling':>15s}")
print("="*70)

---
## Part 8: Failure Mode Analysis

Understanding *why* policies fail is more instructive than knowing their success rate. Let's systematically analyze the failure modes of both policies.

In [ ]:
# Run more evaluations specifically to collect failure cases

def detailed_evaluation(policy, preprocessor, postprocessor, n_episodes=20, max_steps=400, seed=100):
    """
    Run evaluation and collect detailed diagnostics.
    Tracks: per-step rewards, joint trajectories, gripper states.
    """
    env = gym.make(
        "gym_aloha/AlohaTransferCube-v0",
        obs_type="pixels_agent_pos",
        render_mode="rgb_array",
    )

    results = []

    for ep in range(n_episodes):
        obs, info = env.reset(seed=seed + ep)
        policy.reset()

        ep_data = {
            'rewards': [],
            'actions': [],
            'states': [],
            'frames': [],
        }

        for step_i in range(max_steps):
            state = torch.from_numpy(obs['agent_pos']).float().unsqueeze(0).to(device)
            if 'pixels' in obs and isinstance(obs['pixels'], dict):
                img = obs['pixels']['top']
            elif 'pixels' in obs:
                img = obs['pixels']
            else:
                img = obs.get('top', obs.get('image', None))

            image = torch.from_numpy(img).float().permute(2, 0, 1).unsqueeze(0).to(device) / 255.0

            obs_dict = {
                'observation.state': state,
                'observation.images.top': image,
            }
            obs_dict = preprocessor(obs_dict)

            with torch.no_grad():
                action = policy.select_action(obs_dict)

            action = postprocessor(action)
            action_np = action.squeeze(0).cpu().numpy()

            obs, reward, terminated, truncated, info = env.step(action_np)

            ep_data['rewards'].append(reward)
            ep_data['actions'].append(action_np)
            ep_data['states'].append(obs['agent_pos'])
            if step_i % 10 == 0:
                ep_data['frames'].append(env.render())

            if terminated or truncated:
                break

        ep_data['success'] = max(ep_data['rewards']) >= 4.0
        ep_data['max_reward'] = max(ep_data['rewards'])
        ep_data['actions'] = np.array(ep_data['actions'])
        ep_data['states'] = np.array(ep_data['states'])
        results.append(ep_data)

    env.close()
    return results

print("Collecting detailed diagnostics...")
print("\nACT:")
act_policy.eval()
act_results = detailed_evaluation(act_policy, act_preprocessor, act_postprocessor, n_episodes=10)
act_sr_detail = np.mean([r['success'] for r in act_results]) * 100
print(f"  Success rate: {act_sr_detail:.0f}%")

print("\nDiffusion Policy:")
diff_policy_fixed.eval()
diff_results = detailed_evaluation(diff_policy_fixed, diff_preprocessor, diff_postprocessor, n_episodes=10)
diff_sr_detail = np.mean([r['success'] for r in diff_results]) * 100
print(f"  Success rate: {diff_sr_detail:.0f}%")

In [ ]:
# Analyze gripper behavior in successes vs failures

def plot_gripper_analysis(results, policy_name):
    """Plot gripper trajectories for successes vs failures."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    successes = [r for r in results if r['success']]
    failures = [r for r in results if not r['success']]

    # Plot successes
    for r in successes:
        t = np.arange(len(r['actions'])) / 50.0
        axes[0].plot(t, r['actions'][:, 6], 'b-', alpha=0.4, linewidth=1)
        axes[0].plot(t, r['actions'][:, 13], 'r-', alpha=0.4, linewidth=1)
    axes[0].set_title(f'{policy_name}: SUCCESSES ({len(successes)} eps)\nBlue=Left grip, Red=Right grip')
    axes[0].set_xlabel('Time (s)')
    axes[0].set_ylabel('Gripper position')
    axes[0].grid(True, alpha=0.3)

    # Plot failures
    if failures:
        for r in failures:
            t = np.arange(len(r['actions'])) / 50.0
            axes[1].plot(t, r['actions'][:, 6], 'b-', alpha=0.4, linewidth=1)
            axes[1].plot(t, r['actions'][:, 13], 'r-', alpha=0.4, linewidth=1)
        axes[1].set_title(f'{policy_name}: FAILURES ({len(failures)} eps)\nBlue=Left grip, Red=Right grip')
    else:
        axes[1].text(0.5, 0.5, 'No failures!', ha='center', va='center',
                    fontsize=16, transform=axes[1].transAxes)
        axes[1].set_title(f'{policy_name}: FAILURES (0 eps)')
    axes[1].set_xlabel('Time (s)')
    axes[1].set_ylabel('Gripper position')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

plot_gripper_analysis(act_results, "ACT")
plot_gripper_analysis(diff_results, "Diffusion Policy")

print("Look for patterns in the failure cases:")
print("- Do the grippers coordinate properly (left opens as right closes)?")
print("- Is the timing consistent across episodes?")
print("- Are failures due to wrong timing or wrong positioning?")

In [ ]:
# Analyze action smoothness — jittery actions indicate poor learning

def compute_action_jitter(results):
    """Compute the mean absolute difference between consecutive actions."""
    jitters = []
    for r in results:
        diffs = np.abs(np.diff(r['actions'], axis=0))  # [T-1, 14]
        jitters.append(np.mean(diffs, axis=0))         # [14]
    return np.mean(jitters, axis=0)  # [14] mean across episodes

act_jitter = compute_action_jitter(act_results)
diff_jitter = compute_action_jitter(diff_results)

fig, ax = plt.subplots(figsize=(14, 5))

joint_names_short = ['L_w', 'L_sh', 'L_el', 'L_fr', 'L_wa', 'L_wr', 'L_gr',
                     'R_w', 'R_sh', 'R_el', 'R_fr', 'R_wa', 'R_wr', 'R_gr']

x = np.arange(14)
width = 0.35

ax.bar(x - width/2, act_jitter, width, label='ACT', color='steelblue', alpha=0.8)
ax.bar(x + width/2, diff_jitter, width, label='Diffusion', color='coral', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(joint_names_short, rotation=45)
ax.set_ylabel('Mean |Δaction| per step')
ax.set_title('Action Smoothness: Lower = Smoother Motion', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"ACT average jitter:       {np.mean(act_jitter):.4f}")
print(f"Diffusion average jitter: {np.mean(diff_jitter):.4f}")
print(f"\n{'ACT' if np.mean(act_jitter) < np.mean(diff_jitter) else 'Diffusion'} produces smoother actions.")

---
## Part 9: Multimodal Action Distributions in Bimanual Tasks

In Week 1, we showed that Diffusion Policy generates diverse trajectories from the same observation. Let's see if this multimodality matters more in bimanual tasks where there are genuinely multiple valid strategies.

In [ ]:
# Generate multiple action trajectories from the same observation
# using Diffusion Policy (natural multimodality via noise)
# and ACT (multimodality via VAE z sampling)

# Get a fixed observation from the environment
env = gym.make("gym_aloha/AlohaTransferCube-v0", obs_type="pixels_agent_pos", render_mode="rgb_array")
obs, _ = env.reset(seed=42)
env.close()

state = torch.from_numpy(obs['agent_pos']).float().unsqueeze(0).to(device)
if isinstance(obs.get('pixels'), dict):
    img = obs['pixels']['top']
elif 'pixels' in obs:
    img = obs['pixels']
else:
    img = obs.get('top')
image = torch.from_numpy(img).float().permute(2, 0, 1).unsqueeze(0).to(device) / 255.0

# Sample 20 trajectories from each policy
n_samples = 20

# ACT samples (via VAE)
act_trajectories = []
act_policy.eval()
for i in range(n_samples):
    act_policy.reset()  # Reset action queue
    obs_dict = act_preprocessor({'observation.state': state, 'observation.images.top': image})
    with torch.no_grad():
        action = act_policy.select_action(obs_dict)
    action = act_postprocessor(action)
    act_trajectories.append(action.squeeze(0).cpu().numpy())

# Diffusion samples (via noise)
diff_trajectories = []
diff_policy_fixed.eval()
for i in range(n_samples):
    diff_policy_fixed.reset()  # Reset action queue
    obs_dict = diff_preprocessor({'observation.state': state, 'observation.images.top': image})
    with torch.no_grad():
        action = diff_policy_fixed.select_action(obs_dict)
    action = diff_postprocessor(action)
    diff_trajectories.append(action.squeeze(0).cpu().numpy())

print(f"Generated {n_samples} trajectories from each policy")
if len(act_trajectories) > 0:
    print(f"ACT action shape: {act_trajectories[0].shape}")
if len(diff_trajectories) > 0:
    print(f"Diff action shape: {diff_trajectories[0].shape}")

In [ ]:
# Visualize: how diverse are the sampled trajectories?
# Focus on the gripper joints (indices 6 and 13) — the most critical for handoff

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# ACT left gripper diversity
for traj in act_trajectories:
    if len(traj.shape) == 1:
        axes[0, 0].axhline(y=traj[6], alpha=0.3, color='blue')
    else:
        axes[0, 0].plot(traj[:, 6], alpha=0.3, color='blue', linewidth=1)
axes[0, 0].set_title('ACT: Left Gripper Samples', fontsize=12)
axes[0, 0].set_ylabel('Gripper position')
axes[0, 0].grid(True, alpha=0.3)

# ACT right gripper diversity
for traj in act_trajectories:
    if len(traj.shape) == 1:
        axes[0, 1].axhline(y=traj[13], alpha=0.3, color='red')
    else:
        axes[0, 1].plot(traj[:, 13], alpha=0.3, color='red', linewidth=1)
axes[0, 1].set_title('ACT: Right Gripper Samples', fontsize=12)
axes[0, 1].grid(True, alpha=0.3)

# Diffusion left gripper diversity
for traj in diff_trajectories:
    if len(traj.shape) == 1:
        axes[1, 0].axhline(y=traj[6], alpha=0.3, color='blue')
    else:
        axes[1, 0].plot(traj[:, 6], alpha=0.3, color='blue', linewidth=1)
axes[1, 0].set_title('Diffusion: Left Gripper Samples', fontsize=12)
axes[1, 0].set_xlabel('Action step')
axes[1, 0].set_ylabel('Gripper position')
axes[1, 0].grid(True, alpha=0.3)

# Diffusion right gripper diversity
for traj in diff_trajectories:
    if len(traj.shape) == 1:
        axes[1, 1].axhline(y=traj[13], alpha=0.3, color='red')
    else:
        axes[1, 1].plot(traj[:, 13], alpha=0.3, color='red', linewidth=1)
axes[1, 1].set_title('Diffusion: Right Gripper Samples', fontsize=12)
axes[1, 1].set_xlabel('Action step')
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle(f'Action Diversity from Same Observation ({n_samples} samples each)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Quantify diversity
act_arr = np.array([t if len(t.shape) > 1 else t[np.newaxis, :] for t in act_trajectories])
diff_arr = np.array([t if len(t.shape) > 1 else t[np.newaxis, :] for t in diff_trajectories])

print(f"\nAction diversity (std across {n_samples} samples):")
print(f"  ACT mean std:       {act_arr.std(axis=0).mean():.4f}")
print(f"  Diffusion mean std: {diff_arr.std(axis=0).mean():.4f}")

---
## Part 10: The Insertion Task (Optional Challenge)

The insertion task is significantly harder than transfer cube. It requires:
- Sub-millimeter precision for peg-in-hole alignment
- Both arms must stabilize simultaneously
- Tiny errors in approach angle cause complete failure

Try training ACT on insertion and compare success rates.

In [ ]:
# Load the insertion dataset
meta_insertion = LeRobotDatasetMetadata("lerobot/aloha_sim_insertion_human")

print("=== Insertion Dataset ===")
print(f"  Episodes: {meta_insertion.total_episodes}")
print(f"  Total frames: {meta_insertion.total_frames}")
print(f"  FPS: {meta_insertion.fps}")

# Visualize a few frames
dataset_insertion = LeRobotDataset("lerobot/aloha_sim_insertion_human")

# Get frames from episode 0
ins_ep_indices = [i for i in range(min(len(dataset_insertion), 2000))
                  if dataset_insertion[i]['episode_index'].item() == 0]

if ins_ep_indices:
    frame_indices = np.linspace(ins_ep_indices[0], ins_ep_indices[-1], 6, dtype=int)

    fig, axes = plt.subplots(1, 6, figsize=(24, 4))
    for i, idx in enumerate(frame_indices):
        s = dataset_insertion[idx]
        img = s['observation.images.top'].permute(1, 2, 0).numpy()
        if img.max() > 1.0:
            img = img / 255.0
        img = np.clip(img, 0, 1)
        axes[i].imshow(img)
        axes[i].axis('off')
        t_sec = (idx - ins_ep_indices[0]) / meta_insertion.fps
        axes[i].set_title(f't={t_sec:.1f}s', fontsize=10)

    plt.suptitle('Insertion Task: Peg-in-Hole with Bimanual Coordination', fontsize=13)
    plt.tight_layout()
    plt.show()

print("\nInsertion is much harder than transfer cube:")
print("- Requires sub-mm precision for alignment")
print("- Both arms must stabilize the socket and peg simultaneously")
print("- Expect much lower success rates than transfer cube")

---
## Part 11: Key Takeaways & Discussion

### What We Learned This Week

**1. Bimanual manipulation is qualitatively different from single-arm tasks**
- 14D action space (vs 2D for PushT)
- Temporal coordination between arms is critical
- Gripper open/close timing makes or breaks the task

**2. ACT is better suited for ALOHA than Diffusion Policy**
- ACT was designed for ALOHA — it uses long action chunks (100 steps) that capture entire manipulation sequences
- Diffusion Policy's shorter horizon (16 steps) requires frequent re-planning, which can break coordination
- ACT's single-pass inference is much faster

**3. Hyperparameters don't transfer between tasks**
- Diffusion Policy's `crop_shape = [84, 84]` works great for PushT (96×96) but destroys ALOHA (480×640)
- Always inspect what your preprocessing does to the actual data!

**4. Data efficiency matters**
- With only 50 demos, ACT's pretrained backbone and VAE give it an advantage
- Diffusion Policy typically needs more data to match ACT on ALOHA

**5. Multimodality exists but may not be the bottleneck**
- For transfer cube, there's mostly one valid strategy (approach → grasp → lift → hand over)
- Diffusion Policy's multimodal strength shines more when there are genuinely different valid solutions

### Looking Ahead to Week 3: LIBERO

LIBERO introduces **multi-task learning** — one policy handling 10-100 different tasks with language conditioning. This is where generalization becomes the challenge.

---

## Exercises

### Exercise 1: Train on Insertion
Train ACT on `lerobot/aloha_sim_insertion_human`. How does the success rate compare to transfer cube? What are the failure modes?

### Exercise 2: Horizon Ablation for Diffusion Policy
Try increasing Diffusion Policy's `horizon` and `n_action_steps` to match ACT's chunk size:
```python
DiffusionConfig(horizon=100, n_action_steps=100, ...)
```
Does a longer planning horizon help on ALOHA?

### Exercise 3: Temporal Ensemble for ACT
ACT supports temporal ensembling — averaging overlapping action predictions for smoother motion:
```python
ACTConfig(temporal_ensemble_coeff=0.01, ...)
```
Does this improve success rate or action smoothness?

### Exercise 4: Data Augmentation
The crop_shape problem is really a data augmentation problem. Experiment with:
- Different crop sizes: `[200, 300]`, `[300, 400]`, `[440, 560]`
- Random vs center crop: `crop_is_random=True/False`
- No crop at all

Plot success rate vs crop size.

### Exercise 5 (Challenge): Cross-Policy Analysis
Record the internal representations of both policies on the same episodes:
- For ACT: visualize the attention patterns — what does the decoder attend to?
- For Diffusion: visualize the SpatialSoftmax keypoints — where does it look on the 480×640 image?

Do both policies attend to the same parts of the scene?

---

## Summary Table

| Component | ACT | Diffusion Policy |
|-----------|-----|------------------|
| Architecture | Transformer encoder-decoder | 1D U-Net |
| Action prediction | Single forward pass → 100 actions | 100-step denoising → 16 actions |
| Multimodality | VAE latent variable z | Noise initialization |
| Vision backbone | ResNet18 (ImageNet pretrained) | ResNet18 (from scratch) |
| Normalization | MEAN_STD everywhere | MIN_MAX for state/action |
| Best for | Data-efficient bimanual tasks | Tasks with genuine multimodality |
| Critical gotcha | n/a | crop_shape must match image size! |

**Next week**: LIBERO — multi-task learning with 130+ manipulation tasks and language conditioning.